# Pamoka 11 - Agentas į agentą (A2A) protokolas


## Sąranka


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Kas yra A2A protokolas?

**Agentų tarpusavio (A2A) protokolas** yra atviras standartas, leidžiantis DI agentams bendrauti,
atrasti vieni kitus ir bendradarbiauti — net jei jie sukurti skirtinguose karkasuose arba talpinami
skirtingose paslaugose.

Pagrindinės sąvokos:

- **Atradimas** – Agentai paskelbia *agento kortelę*, kuri aprašo jų galimybes, todėl
  kitiems agentams (arba orkestratoriams) lengva rasti tinkamą specialistą užduočiai.
- **Žinučių perdavimas** – Agentai keičiasi struktūrizuotomis žinutėmis per bendrą protokolą, tad
  vieno agento užklausa gali būti suprasta ir įvykdyta kitu, nepaisant jo vidinių
  įgyvendinimo.
- **Užduoties gyvavimo ciklas** – A2A apibrėžia būsenas, tokias kaip *pateikta*, *vykdoma*, *atlikta* ir
  *nepavyko*, suteikdama orkestratoriui pilną matomumą, kaip vyksta deleguotos užduoties eiga.

Šioje pamokoje mes simuliuojame A2A stiliaus bendradarbiavimą sujungdami tris specializuotus kelionių agentus
į darbo srautą, kuriame kiekvienas agentas prisideda savo ekspertize ir perduoda rezultatus kitam.


## Specializuotų kelionių agentų kūrimas


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Daugiagentinis bendradarbiavimas per darbo eigą

Mes sujungiame tris agentus į nuoseklų darbo srautą, kuris atspindi A2A žinučių perdavimą:

1. **CurrencyExchangeAgent** gauna vartotojo užklausą ir pateikia valiutų gaires.
2. **ActivityPlannerAgent** gauna praturtintą kontekstą ir prideda veiklų rekomendacijas.
3. **TravelManagerAgent** sintezuoja abi įvestis į galutinę kelionės santrauką.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Supratimas apie A2A gamyboje

Gamybos aplinkoje A2A protokolas atveria galingas tarp paslaugų scenarijas:

| Galimybė | Aprašymas |
|---|---|
| **Sąveika tarp framework'ų** | Vienu framework'u sukurtas agentas gali deleguoti užduotis agentui, sukurtam bet kuriame kitame A2A suderinamame framework'e, todėl įmanoma tikra tarp organizacijų sąveika. |
| **Paslaugų ribos** | Agentai gali būti atskiruose mikroservisuose, debesų regionuose arba net skirtingose organizacijose, vis tiek sklandžiai bendradarbiaudami. |
| **Dinaminis atradimas** | Orkestratorius gali vykdymo metu užklausti Agent Card registrą, kad rastų geriausiai tinkantį specialistą konkrečiai sub-užduočiai. |
| **Srautinė transliacija ir push pranešimai** | A2A palaiko Server-Sent Events (SSE) realaus laiko pažangos atnaujinimams ir push pranešimus ilgai trunkančioms užduotims. |

Aukščiau sukurta darbo eiga yra supaprastinta, vieno proceso šio modelio versija. Tikroje diegimo aplinkoje kiekvienas agentas pateiktų HTTP galinį tašką, paskelbtų Agent Card ir bendrautų per A2A JSON-RPC protokolą.


## Santrauka

Šioje pamokoje sužinojote:

1. **Kas yra A2A protokolas** — atviras standartas agentų tarpusavio atradimui, pranešimų siuntimui,
   ir užduočių valdymui.
2. **Kaip sukurti specializuotus agentus** — valiutų keitimo agentą, veiklų planuotojo agentą,
   ir kelionių valdymo orkestratorių.
3. **Kaip sujungti agentus į darbo eigą** — naudojant `WorkflowBuilder` modeliuoti nuoseklų
   žinučių perdavimą tarp agentų.
4. **Kaip A2A veikia produkcijoje** — leidžiantis bendradarbiauti tarp skirtingų sistemų ir paslaugų,
   naudojant dinaminį aptikimą ir srautinį atnaujinimą.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, atkreipkite dėmesį, kad automatizuoti vertimai gali turėti klaidų ar netikslumų. Originalų dokumentą jo pradinėje (gimtają) kalboje reikėtų laikyti autoritetingu šaltiniu. Kritiškai svarbiai informacijai rekomenduojamas profesionalus žmogaus atliktas vertimas. Mes neatsakome už bet kokius nesusipratimus ar neteisingas interpretacijas, kylančias dėl šio vertimo naudojimo.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
